## **Deep Learning & LLMs for NLP**

In [ ]:


!pip install torch transformers datasets requests numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.11.0+cpu


PART A: RNN - Character-Level Language Model

In [ ]:
# Load Tiny Shakespeare dataset
import urllib.request

# Fetch directly from Karpathy's original GitHub repo
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = urllib.request.urlopen(url)

# Read, decode, and grab the first 10K characters
text = response.read().decode('utf-8')[:10000]

print(f"Text length: {len(text)} characters")
print(f"Sample: {text[:200]}")

Text length: 10000 characters
Sample: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
# Create character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars[:30])}...")

Vocabulary size: 57
Characters: 
 !',-.:;?ABCDEFHIJLMNOPRSTUVW...


In [ ]:
# Prepare sequences
seq_length = 30
X, y = [], []

for i in range(len(text) - seq_length):
    X.append([char_to_idx[c] for c in text[i:i+seq_length]])
    y.append(char_to_idx[text[i+seq_length]])

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

print(f"Sequences: {X.shape[0]}, Sequence length: {seq_length}")

Sequences: 9970, Sequence length: 30


In [ ]:
# Simple RNN model
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Last timestep
        return out

# Create model
rnn_model = CharRNN(vocab_size, embed_dim=32, hidden_dim=64).to(device)
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")

RNN Parameters: 11,801


**Exercise A.1: Train the RNN**

In [ ]:
# Hyperparameters
batch_size = 128
epochs = 5
learning_rate = 0.001  # A solid default for Adam to avoid overshooting

# DataLoader
dataset = TensorDataset(X[:5000], y[:5000])  # Use subset for speed
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=learning_rate)

# Training loop
losses = []
for epoch in range(epochs):
    total_loss = 0
    for batch_X, batch_y in loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # 1. Zero gradients: Clear old gradients from the last step
        optimizer.zero_grad()

        # 2. Forward pass: Compute predicted outputs by passing inputs to the model
        outputs = rnn_model(batch_X)

        # 3. Compute loss: Calculate the difference between predictions and actual labels
        # Note: Depending on your RNN architecture, you might need to reshape outputs/labels here
        loss = criterion(outputs, batch_y)

        # 4. Backward pass: Compute the gradient of the loss with respect to model parameters
        loss.backward()

        # 5. Update weights: Step the optimizer to adjust parameters
        optimizer.step()

        # Accumulate total loss for this batch to calculate the epoch average
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 2.4803
Epoch 2/5, Loss: 2.4092
Epoch 3/5, Loss: 2.3419
Epoch 4/5, Loss: 2.2967
Epoch 5/5, Loss: 2.2345


**PART B: LSTM - Sentiment Analysis**

In [ ]:

from datasets import load_dataset

# Load IMDB dataset using the explicit namespace
imdb = load_dataset("stanfordnlp/imdb")

# Small sample for quick training
train_texts = imdb['train']['text'][:1000]
train_labels = imdb['train']['label'][:1000]
test_texts = imdb['test']['text'][:200]
test_labels = imdb['test']['label'][:200]

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Train: 1000, Test: 200


In [ ]:
# Simple tokenization and vocabulary
from collections import Counter
import re

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())[:100]  # Max 100 tokens

# Build vocabulary from training data
all_tokens = [tok for text in train_texts for tok in tokenize(text)]
vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(all_tokens).most_common(5000))}
vocab[''] = 0
vocab[''] = 1

print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 5001


In [ ]:
# Encode texts
def encode_text(text, max_len=100):
    tokens = tokenize(text)
    encoded = [vocab.get(t, 1) for t in tokens]  # 1 = UNK
    padded = encoded[:max_len] + [0] * (max_len - len(encoded))
    return padded[:max_len]

X_train = torch.tensor([encode_text(t) for t in train_texts])
y_train = torch.tensor(train_labels)
X_test = torch.tensor([encode_text(t) for t in test_texts])
y_test = torch.tensor(test_labels)

print(f"Train shape: {X_train.shape}")

Train shape: torch.Size([1000, 100])


**Exercise B.1: Complete the LSTM Model**

In [ ]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 1. Define LSTM layer
        # input_size = embed_dim (the size of the vectors coming from the embedding layer)
        # hidden_size = hidden_dim (the size of the LSTM's internal memory)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout(x)

        # 2. Pass through LSTM
        # lstm_out shape: (batch_size, seq_len, hidden_dim)
        # hidden shape: (num_layers, batch_size, hidden_dim)
        lstm_out, (hidden, cell) = self.lstm(x)

        # 3. Use the last hidden state
        # hidden[-1] grabs the hidden state from the final layer of the LSTM
        # This acts as the final "summary" of the entire sequence
        out = self.fc(hidden[-1])

        return out

# Create model
lstm_model = LSTMClassifier(
    vocab_size=len(vocab), # Assuming 'vocab' is defined earlier in your script
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,474


In [ ]:
# Find the absolute maximum token index in your dataset
max_index = max(X_train.max().item(), X_test.max().item())
print(f"Max token index: {max_index}")

# Initialize your model with max_index + 1
lstm_model = LSTMClassifier(
    vocab_size=max_index + 1,  # +1 because 0-indexed!
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

Max token index: 5001


In [ ]:
# Create model with corrected vocab_size
lstm_model = LSTMClassifier(
    vocab_size=5002,  # max_index (5001) + 1
    embed_dim=64,
    hidden_dim=64,
    num_classes=2
).to(device)

print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

LSTM Parameters: 353,538


In [ ]:
# Quick training
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

# Train for 3 epochs
for epoch in range(3):
    lstm_model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = lstm_model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluate
lstm_model.eval()
with torch.no_grad():
    test_output = lstm_model(X_test.to(device))
    preds = torch.argmax(test_output, dim=1).cpu()
    acc = (preds == y_test).float().mean()
    print(f"\nTest Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.4432
Epoch 2, Loss: 0.0025
Epoch 3, Loss: 0.0004

Test Accuracy: 1.0000


**PART C: GRU - News Classification**

In [ ]:
from datasets import load_dataset

# Load AG News using the explicit namespace to satisfy the new HF hub rules
ag_news = load_dataset("fancyzhx/ag_news")
ag_train = ag_news['train'].shuffle(seed=42).select(range(2000))
ag_test = ag_news['test'].shuffle(seed=42).select(range(500))

ag_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
print(f"Classes: {list(ag_labels.values())}")

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Classes: ['World', 'Sports', 'Business', 'Sci/Tech']


In [ ]:
import torch
import torch.nn as nn
from collections import Counter

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # 1. Define GRU layer
        # Exact same signature as nn.LSTM
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)

        # 2. GRU forward pass
        # Returns gru_out: (batch_size, seq_len, hidden_dim)
        # Returns hidden: (num_layers, batch_size, hidden_dim)
        gru_out, hidden = self.gru(x)

        # 3. Use the last hidden state for classification
        out = self.fc(hidden[-1])
        return out

# Build vocabulary and encode
ag_tokens = [tok for item in ag_train for tok in tokenize(item['text'])]
ag_vocab = {word: idx+2 for idx, (word, _) in enumerate(Counter(ag_tokens).most_common(5000))}

# Assign explicit keys for special tokens
ag_vocab['<PAD>'] = 0
ag_vocab['<UNK>'] = 1

def encode_ag(text, vocab, max_len=50):
    tokens = tokenize(text)
    # Use 1 (<UNK>) for any word not found in the vocab
    encoded = [vocab.get(t, 1) for t in tokens]
    return (encoded[:max_len] + [0] * max_len)[:max_len]

X_ag_train = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_train])
y_ag_train = torch.tensor([item['label'] for item in ag_train])
X_ag_test = torch.tensor([encode_ag(item['text'], ag_vocab) for item in ag_test])
y_ag_test = torch.tensor([item['label'] for item in ag_test])

print(f"AG News - Train: {X_ag_train.shape}, Test: {X_ag_test.shape}")

AG News - Train: torch.Size([2000, 50]), Test: torch.Size([500, 50])


In [ ]:
# 1. Initialize the model (Make sure this cell is actually executed!)
gru_model = GRUClassifier(
    vocab_size=len(ag_vocab),
    embed_dim=64,
    hidden_dim=64,
    num_classes=4
).to(device)

print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")

GRU Parameters: 345,348


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1. Create DataLoaders
batch_size = 64
train_dataset = TensorDataset(X_ag_train, y_ag_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 2. Setup Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_model.parameters(), lr=0.001)

# 3. Training Loop
epochs = 5
print("Starting GRU Training...")

for epoch in range(epochs):
    gru_model.train()
    total_loss = 0
    correct = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = gru_model(batch_X)
        loss = criterion(output, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Calculate training accuracy for this batch
        preds = torch.argmax(output, dim=1)
        correct += (preds == batch_y).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / len(train_dataset)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}")

Starting GRU Training...
Epoch 1/5 | Loss: 1.3871 | Accuracy: 0.2650
Epoch 2/5 | Loss: 1.3689 | Accuracy: 0.2890
Epoch 3/5 | Loss: 1.3581 | Accuracy: 0.3165
Epoch 4/5 | Loss: 1.3371 | Accuracy: 0.3365
Epoch 5/5 | Loss: 1.2486 | Accuracy: 0.4195


In [19]:
# Use pre-trained NER model from Hugging Face
from transformers import pipeline

# Load NER pipeline (uses BERT-based model)
print("Loading NER model...")
ner_pipeline = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")
print("Model loaded!")

Loading NER model...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded!


In [20]:
# Example NER
text = "Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO."

entities = ner_pipeline(text)
print(f"Text: {text}\n")
print("Entities found:")
for ent in entities:
    print(f"  {ent['word']:20} -> {ent['entity_group']:10} (score: {ent['score']:.3f})")

Text: Apple Inc. was founded by Steve Jobs in Cupertino, California. Tim Cook is the current CEO.

Entities found:
  Apple Inc            -> ORG        (score: 0.999)
  Steve Jobs           -> PER        (score: 0.903)
  Cupertino            -> LOC        (score: 0.998)
  California           -> LOC        (score: 0.999)
  Tim Cook             -> PER        (score: 1.000)


In [24]:
# 3 sentences covering people, organizations, and locations
my_sentences = [
    "Elon Musk announced that SpaceX will launch a new rocket from Cape Canaveral.",
    "The European Union is headquartered in Brussels and is led by Ursula von der Leyen.",
    "Sundar Pichai visited the Google campus in Mountain View to discuss artificial intelligence."
]

for sent in my_sentences:
    print(f"\nText: {sent}")
    entities = ner_pipeline(sent)
    for ent in entities:
        print(f"  {ent['word']:20} -> {ent['entity_group']}")


Text: Elon Musk announced that SpaceX will launch a new rocket from Cape Canaveral.
  Elon Musk            -> ORG
  SpaceX               -> ORG
  Cape Canaveral       -> LOC

Text: The European Union is headquartered in Brussels and is led by Ursula von der Leyen.
  European Union       -> ORG
  Brussels             -> LOC
  Ursula von der Leye  -> PER

Text: Sundar Pichai visited the Google campus in Mountain View to discuss artificial intelligence.
  Sundar Pichai        -> PER
  Google               -> ORG
  Mountain View        -> LOC


In [25]:
!pip install mistralai opentelemetry-api==1.42.1 opentelemetry-semantic-conventions==0.63b1

In [32]:
MISTRAL_API_KEY = "S6CaoVz6ncQjSteNIRiEiAYHuEB1Rsmk"  # YOUR API KEY HERE

In [33]:
import requests

def query_mistral(prompt, max_tokens=150):
    """Query Mistral API."""
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {MISTRAL_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "mistral-small-latest",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        return f"Error: {response.status_code} - {response.text}"

# Test (only if API key is set)
if MISTRAL_API_KEY != "___":
    response = query_mistral("What is NLP in one sentence?")
    print(f"Mistral: {response}")
else:
    print("Please set your MISTRAL_API_KEY above.")

Mistral: Natural Language Processing (NLP) is a field of AI that enables computers to understand, interpret, and generate human language.


In [34]:
def query_mistral(prompt, max_tokens=10):
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

In [36]:
test_review = "This movie was absolutely terrible. The acting was bad and the plot made no sense."

# LLM approach
if MISTRAL_API_KEY != "S6CaoVz6ncQjSteNIRiEiAYHuEB1Rsmk":
    prompt = f"""Classify the sentiment of this review as 'positive' or 'negative'.
Just respond with one word.

Review: {test_review}

Sentiment:"""

    llm_result = query_mistral(prompt, max_tokens=10)
    print(f"LLM Sentiment: {llm_result}")

# Traditional LSTM approach (if model trained)
try:
    encoded = torch.tensor([encode_text(test_review)]).to(device)
    lstm_model.eval()
    with torch.no_grad():
        lstm_pred = torch.argmax(lstm_model(encoded)).item()
    print(f"LSTM Sentiment: {'positive' if lstm_pred == 1 else 'negative'}")
except:
    print("LSTM model not available")

LSTM Sentiment: negative


In [37]:
from mistralai import Mistral
import os

# Set your valid key here (remove the hardcoded string from your 'if' condition)
MY_KEY = "S6CaoVz6ncQjSteNIRiEiAYHuEB1Rsmk"

# Initialize the client with the actual key
client = Mistral(api_key=MY_KEY)


In [ ]:
# Now you can just run this without the if-check gating your code
long_text = """
Natural language processing (NLP) is a subfield of linguistics, computer science,
and artificial intelligence concerned with the interactions between computers and
human language, in particular how to program computers to process and analyze large
amounts of natural language data. The result is a computer capable of understanding
the contents of documents, including the contextual nuances of the language within them.
"""

summary_prompt = f"Summarize this in one sentence:\n\n{long_text}"
summary = client.chat.complete(
    model="mistral-small-latest",
    messages=[{"role": "user", "content": summary_prompt}]
)

print(f"Summary: {summary.choices[0].message.content}")

Summary: Natural language processing (NLP) is a field at the intersection of linguistics, computer science, and AI that enables computers to process, analyze, and understand human language, including its contextual nuances.


In [ ]:
print(f"LSTM Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
print(f"RNN Parameters: {sum(p.numel() for p in rnn_model.parameters()):,}")
print(f"GRU Parameters: {sum(p.numel() for p in gru_model.parameters()):,}")

LSTM Parameters: 353,538
RNN Parameters: 11,801
GRU Parameters: 345,348


**QUESTION 1 :** L'écart de taille entre ces deux architectures récurrentes provient directement de leur logique de régulation interne : là où le LSTM s'appuie sur une structure lourde à trois portes indépendantes (entrée, oubli, sortie) associées à un état de cellule dédié pour segmenter finement la mémoire, le GRU simplifie ce processus en fusionnant l'état de cellule et l'état caché tout en combinant l'oubli et l'entrée au sein d'une unique porte de mise à jour, ce qui lui permet d'éliminer tout un système de calcul et de réduire ses matrices de poids d'environ un tiers.

**Question 2 :** Le LSTM surclasse le RNN traditionnel sur les longs textes car il résout le problème de la disparition du gradient, un phénomène où les multiplications successives de matrices lors de la rétropropagation à travers le temps réduisent le signal à néant et font perdre au modèle la mémoire du début de la phrase ; grâce à son "autoroute" linéaire d'information (le cell state) régulée par ses portes, le LSTM permet au gradient de circuler librement sur de longues distances temporelles afin de lier facilement les premiers mots d'un avis à sa classification finale.

**Question 3 :** Si les LLMs brillent par leur polyvalence, leur culture générale et leur capacité à saisir le second degré ou à résumer des textes sans réentraînement, ils souffrent de coûts d'API escaladables, d'une latence réseau élevée et de risques importants de fuite de données confidentielles vers des serveurs tiers ; les modèles classiques comme le LSTM ou le GRU restent donc indispensables lorsqu'on recherche une exécution locale ultra-rapide en temps réel, une confidentialité absolue des données traitées en circuit fermé, ou une solution légère et économique pour accomplir une tâche de classification simple et répétitive.